# Extraction et contrôle initial du dataset PaySim

Ce notebook constitue la première étape du projet Data & BI consacré à la détection de fraude aux paiements mobiles.

- **Dataset :** PaySim, disponible sur Kaggle
- **Objectif :** détecter si une transaction est frauduleuse
- **Cible :** `isFraud`
- **Unité d'analyse :** une ligne correspond à une transaction

On récupère les données, on les charge dans pandas et on réalise uniquement des contrôles simples avant la modélisation.

## 1. Installation des dépendances

La cellule suivante peut être exécutée dans Jupyter si les bibliothèques ne sont pas encore installées. Le point d'exclamation utilise l'environnement Python du notebook.

In [2]:
%pip install -q kagglehub pandas

Note: you may need to restart the kernel to use updated packages.


## 2. Imports Python

`kagglehub` télécharge le dataset dans son cache local. `pandas` sert à lire et contrôler le fichier tabulaire. `pathlib` permet de manipuler les chemins de façon portable.

In [3]:
from pathlib import Path

import kagglehub
import pandas as pd

## 3. Téléchargement et identification du fichier CSV

On utilise l'identifiant Kaggle `ealaxi/paysim1`. `dataset_download` renvoie le dossier local du cache KaggleHub ; il n'y a donc pas de chemin propre à une machine écrit en dur dans le notebook.

Si Kaggle demande une authentification, il faut généralement être connecté à Kaggle et disposer d'un accès API configuré (par exemple une variable d'environnement ou un fichier de configuration Kaggle). Les identifiants ne doivent pas être écrits dans le notebook ni commités dans Git.

In [4]:
DATASET_HANDLE = "ealaxi/paysim1"

# KaggleHub télécharge une seule fois puis réutilise son cache local.
dataset_dir = Path(kagglehub.dataset_download(DATASET_HANDLE))
csv_files = sorted(dataset_dir.rglob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"Aucun fichier CSV trouvé dans {dataset_dir}")
if len(csv_files) > 1:
    print("Plusieurs CSV trouvés :")
    for file in csv_files:
        print(f"- {file.relative_to(dataset_dir)}")
    raise RuntimeError("Le fichier CSV doit être sélectionné explicitement.")

csv_path = csv_files[0]
print(f"Fichier sélectionné : {csv_path.name}")
print(f"Dossier KaggleHub : {dataset_dir}")

100%|██████████| 178M/178M [00:36<00:00, 5.16MB/s] 

Extracting files...


Fichier sélectionné : PS_20174392719_1491204439457_log.csv
Dossier KaggleHub : C:\Users\Ayoox\.cache\kagglehub\datasets\ealaxi\paysim1\versions\2


## 4. Chargement dans pandas

Le DataFrame brut s'appelle `df`. On le conserve comme référence et on ne crée pas d'autre copie complète à ce stade.

In [5]:
df = pd.read_csv(csv_path)
print("DataFrame chargé.")

DataFrame chargé.


## 5. Premiers contrôles

Ces commandes permettent de vérifier rapidement le contenu, la taille, les noms de colonnes et les types de données. `df.info()` donne aussi une vue synthétique de la mémoire utilisée et des valeurs non nulles.

In [6]:
display(df.head())
print("Dimensions (lignes, colonnes) :", df.shape)
print("Colonnes :")
print(df.columns.tolist())
print("Types de données :")
display(df.dtypes)
print("Informations générales :")
df.info()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


Dimensions (lignes, colonnes) : (6362620, 11)
Colonnes :
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']
Types de données :


step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object

Informations générales :
<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


## 6. Contrôles qualité simples

On vérifie les valeurs manquantes, les doublons et la distribution de la variable cible. PaySim est fortement déséquilibré : la proportion de fraudes est généralement très faible.

In [7]:
missing_by_column = df.isna().sum().sort_values(ascending=False)
print("Valeurs manquantes par colonne :")
display(missing_by_column.to_frame("missing_count"))

duplicate_count = int(df.duplicated().sum())
print(f"Nombre de lignes dupliquées : {duplicate_count}")

if "isFraud" not in df.columns:
    raise KeyError("La colonne cible 'isFraud' est absente du dataset.")
fraud_distribution = df["isFraud"].value_counts(dropna=False).sort_index()
fraud_proportion = float(df["isFraud"].mean() * 100)
print("Distribution de isFraud :")
display(fraud_distribution.to_frame("count"))
print(f"Proportion de fraudes : {fraud_proportion:.4f} %")

Valeurs manquantes par colonne :


,missing_count
step,0
type,0
amount,0
nameOrig,0
oldbalanceOrg,0
newbalanceOrig,0
nameDest,0
oldbalanceDest,0
newbalanceDest,0
isFraud,0


Nombre de lignes dupliquées : 0
Distribution de isFraud :


,count
isFraud,
0,6354407
1,8213


Proportion de fraudes : 0.1291 %


## 7. Vérification de la granularité

La granularité attendue est **une ligne = une transaction**. La colonne `step` représente une période de temps simulée, mais ne change pas cette unité d'analyse. Les identifiants et les montants décrivent la transaction observée sur cette ligne.

In [8]:
print(f"Une ligne représente une transaction ; le DataFrame contient {len(df):,} transactions.")
print(f"Nombre de valeurs distinctes dans nameOrig : {df['nameOrig'].nunique():,}")
print(f"Nombre de valeurs distinctes dans nameDest : {df['nameDest'].nunique():,}")

Une ligne représente une transaction ; le DataFrame contient 6,362,620 transactions.
Nombre de valeurs distinctes dans nameOrig : 6,353,307
Nombre de valeurs distinctes dans nameDest : 2,722,362


## 8. Colonnes à risque de fuite de données (data leakage)

Certaines colonnes peuvent donner au modèle une information disponible seulement après la transaction, ou contenir le résultat d'une règle déjà existante :

- `newbalanceOrig` et `newbalanceDest` correspondent aux soldes **après** la transaction ; leur disponibilité réelle au moment de la prédiction doit être vérifiée.
- `isFlaggedFraud` est la sortie d'une règle existante. L'utiliser comme variable explicative classique peut faire apprendre au modèle cette règle plutôt que le comportement frauduleux.

Ces colonnes ne sont pas supprimées automatiquement de `df` : il faut conserver le jeu brut et documenter la décision de modélisation.

In [9]:
potential_leakage_columns = [
    column for column in ["newbalanceOrig", "newbalanceDest", "isFlaggedFraud"]
    if column in df.columns
]
print("Colonnes potentiellement problématiques :", potential_leakage_columns)

Colonnes potentiellement problématiques : ['newbalanceOrig', 'newbalanceDest', 'isFlaggedFraud']


## 9. Copie préparée pour la suite

Pour éviter d'utiliser par inadvertance des informations postérieures à la transaction, on crée `df_model` en retirant les soldes après transaction. Le choix concernant `isFlaggedFraud` reste explicite : cette colonne est retirée car elle représente déjà une règle de détection existante, et non une information métier indépendante.

`df` reste inchangé et constitue la source brute. La copie est créée une seule fois pour la suite du projet.

In [10]:
columns_to_remove_for_model = [
    column for column in ["newbalanceOrig", "newbalanceDest", "isFlaggedFraud"]
    if column in df.columns
]

df_model = df.drop(columns=columns_to_remove_for_model)
print("Colonnes retirées de df_model :", columns_to_remove_for_model)
print("Dimensions de df_model :", df_model.shape)

Colonnes retirées de df_model : ['newbalanceOrig', 'newbalanceDest', 'isFlaggedFraud']
Dimensions de df_model : (6362620, 8)


## 10. Résumé des contrôles

Ce résumé regroupe les principaux résultats obtenus à partir du DataFrame brut `df`.

In [11]:
summary = pd.Series({
    "Nombre de lignes": len(df),
    "Nombre de colonnes": df.shape[1],
    "Nombre de fraudes": int(df["isFraud"].sum()),
    "Proportion de fraudes (%)": round(fraud_proportion, 4),
    "Nombre de doublons": duplicate_count,
    "Nombre total de valeurs manquantes": int(df.isna().sum().sum()),
    "Colonnes potentiellement problématiques": ", ".join(potential_leakage_columns)
})
display(summary.to_frame("Valeur"))

,Valeur
Nombre de lignes,6362620
Nombre de colonnes,11
Nombre de fraudes,8213
Proportion de fraudes (%),0.1291
Nombre de doublons,0
Nombre total de valeurs manquantes,0
Colonnes potentiellement problématiques,"newbalanceOrig, newbalanceDest, isFlaggedFraud"


### À vérifier avant de considérer le notebook comme terminé

- vérifier que l'authentification Kaggle fonctionne dans l'environnement utilisé ;
- vérifier que le dataset téléchargé est bien la version attendue et que le CSV trouvé est unique ;
- vérifier que `isFraud` est bien binaire et ne contient pas de valeurs inattendues ;
- confirmer avec le contexte métier quelles colonnes sont réellement disponibles au moment de la prédiction ;
- décider, dans le prochain notebook, comment traiter les doublons et les éventuelles valeurs manquantes ;
- relancer le notebook de haut en bas dans un environnement propre pour confirmer sa reproductibilité.